In [19]:
import torch
import lightning.pytorch as pl
import numpy as np
import os
from typing import Optional, Union

from models.vqvae import VQVAE
from models.resnet import ResNet

class ForecastDataset(torch.utils.data.Dataset):

    def __init__(self, path: str, vqvae: VQVAE, channels: int, img_res: tuple[int, int]) -> None:
        super(ForecastDataset, self).__init__()

        self.vqvae = vqvae
        self.device = vqvae.device
        self.path = path

        self.channels = channels
        self.img_res = img_res

        self.data = self._load_data()

    def _load_data(self):
        path = self.path
        c = self.channels
        w, h = self.img_res

        data = np.memmap(path, mode="r", dtype=np.float32).reshape(-1, c, w, h)

        return data

    def __len__(self):
        return self.data.shape[0] - 1

    @torch.no_grad()
    def __getitem__(self, idx):
        x1 = self.data[idx]
        x2 = self.data[idx + 1]

        x1 = torch.tensor(x1, dtype=torch.float32).to(self.device)
        x2 = torch.tensor(x2, dtype=torch.float32).to(self.device)

        _, latent_x1, _ = self.vqvae(x1.unsqueeze(0))
        _, latent_x2, _ = self.vqvae(x2.unsqueeze(0))

        latent_x1 = latent_x1.squeeze(0)
        latent_x2 = latent_x2.squeeze(0)

        return latent_x1, latent_x2
    


In [22]:
### Train ###

# Hyperparameters
lr = 3e-4
batch_size = 4
epochs = 10

# Data
path = "train.memmap"
channels = 5
img_res = (128, 64)

# Model
vqvae = VQVAE(in_channels=channels)
resnet = ResNet(in_channels=8)

# Dataset
dataset = ForecastDataset(path, vqvae, channels, img_res)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Optimizer
optimizer = torch.optim.Adam(resnet.parameters(), lr=lr)
criteria = torch.nn.functional.mse_loss


# Train Loop
for epoch in range(epochs):
    for x1, x2 in dataloader:
        pred = resnet(x1)

        pred_quantized, quantize_loss, _ = vqvae.quantize(pred)

        prediction_loss = criteria(pred, x2)
        commitment_loss = quantize_loss["commitment_loss"]

        loss = prediction_loss + commitment_loss
        
        print(f"Prediction Loss: {prediction_loss}, Commitment Loss: {commitment_loss}")

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

Prediction Loss: 0.8647423982620239, Commitment Loss: 0.18404893577098846
Prediction Loss: 0.7474907040596008, Commitment Loss: 0.1634444147348404
Prediction Loss: 0.6264973878860474, Commitment Loss: 0.1469259411096573
Prediction Loss: 0.5533750653266907, Commitment Loss: 0.14134033024311066
Prediction Loss: 0.4848076105117798, Commitment Loss: 0.1372707337141037
Prediction Loss: 0.4209524691104889, Commitment Loss: 0.12951143085956573
Prediction Loss: 0.366083562374115, Commitment Loss: 0.12364719808101654
Prediction Loss: 0.34566131234169006, Commitment Loss: 0.12138017266988754
Prediction Loss: 0.31615975499153137, Commitment Loss: 0.12091987580060959
Prediction Loss: 0.27225247025489807, Commitment Loss: 0.11683855950832367
Prediction Loss: 0.2670398950576782, Commitment Loss: 0.11405901610851288
Prediction Loss: 0.21866652369499207, Commitment Loss: 0.10593792796134949
Prediction Loss: 0.21052923798561096, Commitment Loss: 0.10450857132673264
Prediction Loss: 0.18427926301956177,

KeyboardInterrupt: 